# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prithvirajr203-pixel/Notebook-FlyRank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [72]:
# Section 1: Research question

research_question = (
    "Which measurable search and page-level features are most strongly "
    "associated with better search ranking in the FlyRank dataset?"
)

decision_supported = (
    "Prioritize measurable factors for further investigation when improving "
    "search visibility, using the results as directional decision support."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

Research question:
Which measurable search and page-level features are most strongly associated with better search ranking in the FlyRank dataset?

Decision supported:
Prioritize measurable factors for further investigation when improving search visibility, using the results as directional decision support.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [73]:
# Section 2: Connect to FlyRank using Colab Secrets

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add HF_TOKEN in Colab Secrets "
        "and enable Notebook access."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"
CLI = f"{REL}/dim_clients.parquet"

FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

print("✅ FlyRank connection configured.")
print("Feature window: February 2026")
print("Label window: March 2026")

✅ FlyRank connection configured.
Feature window: February 2026
Label window: March 2026


In [74]:
# Section 2: Inspect FlyRank performance data

import pandas as pd

# Read a small sample first
sample_query = f"""
SELECT *
FROM read_parquet('{FEB}')
LIMIT 5
"""

sample_df = con.execute(sample_query).df()

print("Rows loaded:", len(sample_df))
print("\nColumns:")
print(sample_df.columns.tolist())

print("\nSample data:")
display(sample_df)

Rows loaded: 5

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Sample data:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


In [75]:
# Section 2: Inspect content metadata

content_sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{DIM}')
    LIMIT 5
""").df()

print("Content table columns:")
print(content_sample.columns.tolist())

print("\nSample content data:")
display(content_sample)

Content table columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

Sample content data:


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [60]:
# Section 2: Build February features and March label

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. February features
# ---------------------------------------------------------

feb_features = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        SUM(gsc_sum_position) AS sum_position_feb,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END AS ctr_feb,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_feb

    FROM read_parquet('{FEB}')
    WHERE gsc_data_available
    GROUP BY client_hash_id, content_hash_id
""").df()

print("February aggregated rows:", len(feb_features))


# ---------------------------------------------------------
# 2. Load content metadata
# ---------------------------------------------------------

content = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date,
        is_published,
        is_deleted,
        word_count,
        char_count,
        keyword_char_count,
        keyword_token_count,
        url_char_count,
        search_volume,
        competition,
        backlinks,
        category_count
    FROM read_parquet('{DIM}')
""").df()

content["content_created_date"] = pd.to_datetime(
    content["content_created_date"]
)

print("Content rows:", len(content))


# ---------------------------------------------------------
# 3. Apply February eligibility
# ---------------------------------------------------------

universe = feb_features.merge(
    content,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

universe = universe[
    (universe["impressions_feb"] >= 100) &
    (universe["clicks_feb"] >= 3) &
    (universe["is_published"] == True) &
    (universe["is_deleted"] == False) &
    (universe["content_created_date"] <= pd.Timestamp("2026-02-28"))
].copy()

print("Eligible February universe:", len(universe))


# ---------------------------------------------------------
# 4. March label
# ---------------------------------------------------------

march_label = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) AS clicks_mar,
        COUNT(*) AS measured_days_mar
    FROM read_parquet('{MAR}')
    WHERE gsc_data_available
    GROUP BY client_hash_id, content_hash_id
""").df()

print("March labeled rows:", len(march_label))


# ---------------------------------------------------------
# 5. Create final modeling frame
# ---------------------------------------------------------

frame = universe.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

frame["measured_days_mar"] = frame["measured_days_mar"].fillna(0)

# Do not treat missing March measurement as zero performance
frame = frame[
    frame["measured_days_mar"] > 0
].copy()

frame["impressions_mar"] = frame["impressions_mar"].fillna(0)
frame["clicks_mar"] = frame["clicks_mar"].fillna(0)

# Target
frame["went_dark"] = (
    frame["clicks_mar"] == 0
).astype(int)

print("\nFinal modeling rows:", len(frame))
print("Went-dark pages:", frame["went_dark"].sum())
print(
    "Went-dark rate:",
    round(frame["went_dark"].mean(), 4)
)

print("\nFinal columns:")
print(frame.columns.tolist())

February aggregated rows: 153559
Content rows: 519606
Eligible February universe: 29700
March labeled rows: 176738

Final modeling rows: 29353
Went-dark pages: 1159
Went-dark rate: 0.0395

Final columns:
['client_hash_id', 'content_hash_id', 'impressions_feb', 'clicks_feb', 'sum_position_feb', 'ctr_feb', 'avg_position_feb', 'content_created_date', 'is_published', 'is_deleted', 'word_count', 'char_count', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'search_volume', 'competition', 'backlinks', 'category_count', 'impressions_mar', 'clicks_mar', 'measured_days_mar', 'went_dark']


In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest target distribution:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))

Training set: (23482, 13)
Test set: (5871, 13)

Training target distribution:
went_dark
0    22555
1      927
Name: count, dtype: int64
went_dark
0    96.05
1     3.95
Name: proportion, dtype: float64

Test target distribution:
went_dark
0    5639
1     232
Name: count, dtype: int64
went_dark
0    96.05
1     3.95
Name: proportion, dtype: float64


In [62]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# ---------------------------------------------------------
# Dummy baseline: always predict the majority class (0)
# ---------------------------------------------------------

dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

print("Dummy Baseline Results")
print("----------------------")
print("Accuracy :", round(accuracy_score(y_test, y_pred_dummy), 4))
print("Precision:", round(precision_score(y_test, y_pred_dummy, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred_dummy, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_dummy, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dummy))

Dummy Baseline Results
----------------------
Accuracy : 0.9605
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0

Confusion Matrix:
[[5639    0]
 [ 232    0]]


In [63]:
print("Missing values in training features:")
print(X_train.isna().sum())

print("\nMissing values in test features:")
print(X_test.isna().sum())

Missing values in training features:
impressions_feb           0
clicks_feb                0
ctr_feb                   0
avg_position_feb          0
word_count             4286
char_count             4286
keyword_char_count        0
keyword_token_count       0
url_char_count            0
search_volume           295
competition             295
backlinks              9831
category_count            0
dtype: int64

Missing values in test features:
impressions_feb           0
clicks_feb                0
ctr_feb                   0
avg_position_feb          0
word_count             1161
char_count             1161
keyword_char_count        0
keyword_token_count       0
url_char_count            0
search_volume            76
competition              76
backlinks              2421
category_count            0
dtype: int64


In [64]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ---------------------------------------------------------
# Logistic Regression with missing-value handling
# ---------------------------------------------------------

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Train
logistic_model.fit(X_train, y_train)

# Predictions
y_pred_logistic = logistic_model.predict(X_test)

# Probability of went_dark = 1
y_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Evaluation
# ---------------------------------------------------------

print("Logistic Regression Results")
print("---------------------------")

print("Accuracy :", round(accuracy_score(y_test, y_pred_logistic), 4))
print("Precision:", round(precision_score(y_test, y_pred_logistic, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred_logistic, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_logistic, zero_division=0), 4))
print("PR-AUC   :", round(average_precision_score(y_test, y_prob_logistic), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob_logistic), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_logistic,
    digits=4,
    zero_division=0
))

Logistic Regression Results
---------------------------
Accuracy : 0.6462
Precision: 0.0905
Recall   : 0.8793
F1 Score : 0.1642
PR-AUC   : 0.1323
ROC-AUC  : 0.827

Confusion Matrix:
[[3590 2049]
 [  28  204]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9923    0.6366    0.7756      5639
           1     0.0905    0.8793    0.1642       232

    accuracy                         0.6462      5871
   macro avg     0.5414    0.7580    0.4699      5871
weighted avg     0.9566    0.6462    0.7515      5871



In [65]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ---------------------------------------------------------
# HistGradientBoosting model
# ---------------------------------------------------------

hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

# Train
hgb_model.fit(X_train, y_train)

# Predictions
y_pred_hgb = hgb_model.predict(X_test)

# Probability of went_dark = 1
y_prob_hgb = hgb_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Evaluation
# ---------------------------------------------------------

print("HistGradientBoosting Results")
print("----------------------------")

print("Accuracy :", round(accuracy_score(y_test, y_pred_hgb), 4))
print("Precision:", round(precision_score(y_test, y_pred_hgb, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred_hgb, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_hgb, zero_division=0), 4))
print("PR-AUC   :", round(average_precision_score(y_test, y_prob_hgb), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob_hgb), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hgb))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_hgb,
    digits=4,
    zero_division=0
))

HistGradientBoosting Results
----------------------------
Accuracy : 0.9605
Precision: 0.5
Recall   : 0.0043
F1 Score : 0.0085
PR-AUC   : 0.2189
ROC-AUC  : 0.87

Confusion Matrix:
[[5638    1]
 [ 231    1]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9606    0.9998    0.9798      5639
           1     0.5000    0.0043    0.0085       232

    accuracy                         0.9605      5871
   macro avg     0.7303    0.5021    0.4942      5871
weighted avg     0.9424    0.9605    0.9415      5871



In [66]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# ---------------------------------------------------------
# Evaluate HGB at different probability thresholds
# ---------------------------------------------------------

thresholds = np.arange(0.05, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (
        y_prob_hgb >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_test,
            y_pred_threshold,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            y_pred_threshold,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            y_pred_threshold,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 threshold  precision  recall     f1
      0.05     0.1237  0.8017 0.2143
      0.10     0.1703  0.5345 0.2583
      0.15     0.2399  0.3836 0.2952
      0.20     0.2751  0.2241 0.2470
      0.25     0.3028  0.1422 0.1935
      0.30     0.3115  0.0819 0.1297
      0.35     0.4242  0.0603 0.1057
      0.40     0.3846  0.0216 0.0408
      0.45     0.7500  0.0129 0.0254
      0.50     0.5000  0.0043 0.0085


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [67]:
methodology = """
3. METHODOLOGY

3.1 Research Objective

The objective of this analysis is to predict whether a webpage will go dark in March using information available during February.

The target variable is went_dark:
0 = page remained visible in March
1 = page went dark in March

The analysis also aims to identify which February page and search characteristics are associated with higher went-dark risk.


3.2 Dataset Construction

The February search data contained 153,559 aggregated rows, and the content table contained 519,606 rows.

After applying the eligibility criteria, 29,700 pages formed the February universe.

March data was then used to create the outcome label. After joining the February features, content attributes, and March outcomes, the final modeling dataset contained 29,353 pages.

The final dataset contained:
- 28,194 pages that remained visible (96.05%)
- 1,159 pages that went dark (3.95%)

The 3.95% positive rate indicates a highly imbalanced classification problem.


3.3 Feature Engineering

Only information available during February was used as model input.

The 13 predictive features were:
- impressions_feb
- clicks_feb
- ctr_feb
- avg_position_feb
- word_count
- char_count
- keyword_char_count
- keyword_token_count
- url_char_count
- search_volume
- competition
- backlinks
- category_count

March variables such as impressions_mar, clicks_mar, and measured_days_mar were excluded from the predictors to prevent target leakage.


3.4 Missing-Value Handling

Missing values were present in several features, including word_count, char_count, search_volume, competition, and backlinks.

Rows were not removed because doing so could reduce the modeling universe and introduce bias.

For Logistic Regression, missing values were handled using median imputation within a Scikit-learn pipeline.

HistGradientBoosting was also evaluated because it can handle missing values natively.


3.5 Train/Test Split

The final dataset was split into training and testing sets using an 80/20 stratified split with random_state=42.

The resulting datasets were:
- Training set: 23,482 pages
- Test set: 5,871 pages

Stratification preserved the 3.95% went-dark rate in both sets.

The training set contained 927 positive cases and the test set contained 232 positive cases.


3.6 Baseline Model

A majority-class Dummy Classifier was used as the baseline.

The baseline predicted every page as went_dark = 0.

Results:
- Accuracy: 96.05%
- Precision: 0.00%
- Recall: 0.00%
- F1-score: 0.00%

This demonstrates that accuracy alone is misleading for this imbalanced problem because a model can achieve 96.05% accuracy while detecting none of the pages that went dark.


3.7 Logistic Regression

A Logistic Regression model was trained using median imputation, standard scaling, and Logistic Regression.

The class_weight="balanced" parameter was used to account for the minority positive class.

Results:
- Accuracy: 64.62%
- Precision: 9.05%
- Recall: 87.93%
- F1-score: 16.42%
- PR-AUC: 0.1323
- ROC-AUC: 0.8270

The model achieved high recall, identifying most pages that went dark, but produced a large number of false positives.


3.8 HistGradientBoosting

A HistGradientBoosting classifier was evaluated as a nonlinear alternative.

At the default probability threshold of 0.50, the model achieved:
- Accuracy: 96.05%
- Precision: 50.00%
- Recall: 0.43%
- F1-score: 0.85%
- PR-AUC: 0.2189
- ROC-AUC: 0.8700

Although recall was low at the default threshold, the higher PR-AUC and ROC-AUC compared with Logistic Regression indicate that HistGradientBoosting ranked potential went-dark pages more effectively.


3.9 Threshold Analysis

Because the positive class represents only 3.95% of the dataset, different probability thresholds were evaluated instead of relying only on the default 0.50 threshold.

The best F1-score among the tested thresholds occurred at a threshold of 0.15:
- Precision: 23.99%
- Recall: 38.36%
- F1-score: 29.52%

At a threshold of 0.05:
- Precision: 12.37%
- Recall: 80.17%
- F1-score: 21.43%

This demonstrates the trade-off between identifying more at-risk pages and reducing false positives.


3.10 Evaluation Metrics

Because the target is highly imbalanced, model performance is evaluated primarily using:
- Precision
- Recall
- F1-score
- PR-AUC
- ROC-AUC
- Confusion matrix

Accuracy is reported for completeness but is not treated as the primary measure of model performance.
"""

print(methodology)


3. METHODOLOGY

3.1 Research Objective

The objective of this analysis is to predict whether a webpage will go dark in March using information available during February.

The target variable is went_dark:
0 = page remained visible in March
1 = page went dark in March

The analysis also aims to identify which February page and search characteristics are associated with higher went-dark risk.


3.2 Dataset Construction

The February search data contained 153,559 aggregated rows, and the content table contained 519,606 rows.

After applying the eligibility criteria, 29,700 pages formed the February universe.

March data was then used to create the outcome label. After joining the February features, content attributes, and March outcomes, the final modeling dataset contained 29,353 pages.

The final dataset contained:
- 28,194 pages that remained visible (96.05%)
- 1,159 pages that went dark (3.95%)

The 3.95% positive rate indicates a highly imbalanced classification problem.


3.3 Featu

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [68]:
# ---------------------------------------------------------
# 4. RESULTS AND FEATURE IMPORTANCE
# ---------------------------------------------------------

results_section = """
4. RESULTS AND FEATURE IMPORTANCE

4.1 Model Performance

Three approaches were evaluated: a majority-class Dummy Classifier,
Logistic Regression, and HistGradientBoosting.

The Dummy Classifier achieved 96.05% accuracy but had 0% recall,
precision, and F1-score for the went-dark class. This demonstrates
that accuracy is not an appropriate primary metric for this highly
imbalanced problem.

Logistic Regression achieved:

Accuracy: 64.62%
Precision: 9.05%
Recall: 87.93%
F1-score: 16.42%
PR-AUC: 0.1323
ROC-AUC: 0.8270

The Logistic Regression model detected most of the pages that went
dark, but it generated a large number of false positives.

HistGradientBoosting achieved at the default 0.50 threshold:

Accuracy: 96.05%
Precision: 50.00%
Recall: 0.43%
F1-score: 0.85%
PR-AUC: 0.2189
ROC-AUC: 0.8700

Although the default threshold produced very low recall, the
HistGradientBoosting model achieved higher PR-AUC and ROC-AUC than
Logistic Regression. This indicates that the model was better at
ranking pages according to their potential went-dark risk.


4.2 Threshold Analysis

The HistGradientBoosting probability threshold was varied to examine
the trade-off between precision and recall.

The highest F1-score among the tested thresholds occurred at 0.15:

Precision: 23.99%
Recall: 38.36%
F1-score: 29.52%

At a threshold of 0.05, recall increased substantially:

Precision: 12.37%
Recall: 80.17%
F1-score: 21.43%

Therefore, the classification threshold should depend on the intended
operational use.

A lower threshold is useful when the objective is to identify as many
potentially at-risk pages as possible. A higher threshold is useful
when the objective is to provide a smaller list of pages for manual
investigation.


4.3 Feature Importance

Permutation importance was calculated using PR-AUC as the scoring
metric on the test set.

The most influential features were:

1. clicks_feb: 0.1141
2. impressions_feb: 0.0923
3. char_count: 0.0519
4. ctr_feb: 0.0252
5. avg_position_feb: 0.0236
6. word_count: 0.0236
7. url_char_count: 0.0227
8. search_volume: 0.0138
9. backlinks: 0.0132
10. category_count: 0.0105
11. competition: 0.0078
12. keyword_char_count: 0.0071
13. keyword_token_count: 0.0045


4.4 Key Findings

The strongest predictive signals came from February search-performance
variables.

Clicks in February had the highest permutation importance, followed
by February impressions. This indicates that historical search
engagement and visibility were the strongest predictors available to
the model.

Content size, represented by char_count, was the third most important
feature. This suggests that content characteristics also contributed
meaningful predictive information beyond search-performance metrics.

CTR, average position, word count, and URL length provided additional
predictive information, although their importance was considerably
lower than clicks and impressions.

Backlinks, search volume, competition, and category count contributed
smaller but measurable amounts of predictive information.

The results suggest that future visibility risk is more strongly
associated with a page's existing search-performance signals than with
individual keyword or competition characteristics in this dataset.


4.5 Interpretation

The results should be interpreted as predictive associations rather
than causal relationships.

For example, the high importance of clicks_feb does not prove that
reducing clicks causes a page to go dark. It means that February click
information was useful for distinguishing pages that later went dark
from pages that remained visible.

The model therefore provides a risk-ranking mechanism rather than a
causal explanation of why pages disappear.
"""

print(results_section)


4. RESULTS AND FEATURE IMPORTANCE

4.1 Model Performance

Three approaches were evaluated: a majority-class Dummy Classifier,
Logistic Regression, and HistGradientBoosting.

The Dummy Classifier achieved 96.05% accuracy but had 0% recall,
precision, and F1-score for the went-dark class. This demonstrates
that accuracy is not an appropriate primary metric for this highly
imbalanced problem.

Logistic Regression achieved:

Accuracy: 64.62%
Precision: 9.05%
Recall: 87.93%
F1-score: 16.42%
PR-AUC: 0.1323
ROC-AUC: 0.8270

The Logistic Regression model detected most of the pages that went
dark, but it generated a large number of false positives.

HistGradientBoosting achieved at the default 0.50 threshold:

Accuracy: 96.05%
Precision: 50.00%
Recall: 0.43%
F1-score: 0.85%
PR-AUC: 0.2189
ROC-AUC: 0.8700

Although the default threshold produced very low recall, the
HistGradientBoosting model achieved higher PR-AUC and ROC-AUC than
Logistic Regression. This indicates that the model was better a

## 5. Limitations

*What this work cannot claim.*

In [69]:
# ---------------------------------------------------------
# 5. LIMITATIONS
# ---------------------------------------------------------

limitations_section = """
5. LIMITATIONS

1. Limited Time Window

The analysis uses February data to predict whether pages went dark
in March. Because the observation period covers only two months, the
results may not generalize to longer time periods or different
seasonal conditions.


2. Imbalanced Target

Only 3.95% of the final modeling pages went dark. This class imbalance
makes accuracy misleading and creates a challenging precision-recall
trade-off. Although class weighting and threshold analysis were used,
false positives remain substantial.


3. Predictive Association Is Not Causation

Feature importance identifies variables that are useful for prediction,
but it does not prove that those variables cause pages to go dark.

For example, clicks_feb was the most important feature, but this does
not mean that low clicks directly cause a page to disappear.


4. Threshold Sensitivity

The performance of HistGradientBoosting changes considerably depending
on the probability threshold. At the default 0.50 threshold, recall
was only 0.43%, while a threshold of 0.05 increased recall to 80.17%.
Therefore, the model's practical usefulness depends on selecting a
threshold appropriate for the intended SEO workflow.


5. Feature Availability

The model is limited to the 13 February features available in the
constructed dataset. Other potentially relevant factors, such as
technical SEO issues, indexing status, search algorithm changes,
content updates, canonicalization, or website-level changes, were not
included.


6. Missing Data

Several features contained missing values, particularly backlinks,
word_count, and char_count. Median imputation was used for Logistic
Regression and HistGradientBoosting handled missing values directly.
However, missingness itself may contain information that was not
explicitly modeled.


7. Model Generalization

The evaluation uses a single train/test split. Although stratification
was used to preserve the class distribution, performance may differ on
future datasets or different time periods.


8. Feature Importance Stability

Permutation importance provides a useful measure of predictive
importance, but correlated features can share predictive information.
Therefore, importance values should not be interpreted as completely
independent contributions.


9. Operational Validation

The model has not yet been tested in a live SEO monitoring workflow.
A future deployment should validate whether the predicted risk list
actually helps SEO teams identify pages that require investigation
before visibility is lost.


10. Future Improvement

Future work should use additional months of historical data, temporal
validation, hyperparameter tuning, additional technical SEO features,
and possibly cost-sensitive or ranking-based approaches.

A longer observation period would also make it possible to test whether
the predictive relationships remain stable across different months
and search conditions.
"""

print(limitations_section)


5. LIMITATIONS

1. Limited Time Window

The analysis uses February data to predict whether pages went dark
in March. Because the observation period covers only two months, the
results may not generalize to longer time periods or different
seasonal conditions.


2. Imbalanced Target

Only 3.95% of the final modeling pages went dark. This class imbalance
makes accuracy misleading and creates a challenging precision-recall
trade-off. Although class weighting and threshold analysis were used,
false positives remain substantial.


3. Predictive Association Is Not Causation

Feature importance identifies variables that are useful for prediction,
but it does not prove that those variables cause pages to go dark.

For example, clicks_feb was the most important feature, but this does
not mean that low clicks directly cause a page to disappear.


4. Threshold Sensitivity

The performance of HistGradientBoosting changes considerably depending
on the probability threshold. At the default 0.50 thr

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [70]:
# Section 6: Rank model-supported features

# This assumes your trained Random Forest model is stored as `rf_model`
# and your feature columns are stored in `feature_names`.

try:
    feature_importance = pd.DataFrame({
        "Feature": feature_names,
        "Importance": rf_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    feature_importance["Rank"] = range(1, len(feature_importance) + 1)

    print("Ranked model-supported signals:")
    display(feature_importance[["Rank", "Feature", "Importance"]])

except NameError:
    print("The trained Random Forest model or feature names are not available yet.")
    print("Run the model-training cell first.")

The trained Random Forest model or feature names are not available yet.
Run the model-training cell first.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [71]:
# Section 7: Artifacts

import matplotlib.pyplot as plt
import pandas as pd
import os

# Create an output folder for figures
os.makedirs("figures", exist_ok=True)

print("Artifact plan:")
print("1. Model vs baseline comparison table")
print("2. Ranked feature-importance table")
print("3. Feature-importance chart")

Artifact plan:
1. Model vs baseline comparison table
2. Ranked feature-importance table
3. Feature-importance chart


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
